# Comprendre les LLM et le RAG avec une mini-FAQ

Ce notebook construit un tout petit assistant qui retrouve des passages dans trois notes, puis prépare une question enrichie de ces passages. Il n'a besoin ni de clé API ni de téléchargement.

À la fin, tu pourras copier le prompt produit dans un LLM que tu utilises déjà et comparer sa réponse avec les sources. Le notebook ne prétend pas générer lui-même une réponse de LLM.

## 1. Qu'est-ce qu'un LLM ?

Un grand modèle de langage (LLM) est un modèle entraîné sur beaucoup de textes. À partir d'un texte en entrée, il prédit progressivement les prochains éléments (les *tokens*) pour produire une suite cohérente.

Il sait bien reformuler, résumer et suivre des consignes, mais une réponse plausible n'est pas forcément vraie. Un LLM n'est pas automatiquement connecté à tes documents ni à Internet.

Dans ce projet, le LLM sera le composant qui rédige la réponse. La recherche des sources sera une étape distincte.

## 2. Qu'est-ce que le RAG ?

RAG signifie *Retrieval-Augmented Generation*, ou génération augmentée par recherche. Le programme cherche d'abord des passages utiles (*Retrieval*), les ajoute à la question (*Augmented*), puis un LLM rédige une réponse (*Generation*).

Le RAG ne réentraîne pas le modèle. Il lui fournit des informations au moment de la question. Si la recherche trouve un mauvais passage, la réponse peut quand même être mauvaise.

## 3. Préparer trois documents

Nos documents sont de courtes notes fictives. Dans un vrai projet, ils pourraient être des pages de documentation, des notes de cours ou des informations de CV que tu as choisi d'utiliser.

In [1]:
documents = [
    {
        "source": "note_llm",
        "text": (
            "Un LLM est un modèle entraîné à prédire la suite d'un texte. "
            "Il peut produire une réponse plausible qui contient pourtant une erreur."
        ),
    },
    {
        "source": "note_rag",
        "text": (
            "Le RAG recherche des passages dans des documents et les ajoute au contexte "
            "envoyé au LLM. Le RAG ne réentraîne pas le modèle."
        ),
    },
    {
        "source": "note_embeddings",
        "text": (
            "Un embedding représente un texte par une liste de nombres. "
            "On peut comparer ces représentations pour retrouver des textes proches par le sens."
        ),
    },
]

for document in documents:
    print(f"[{document['source']}] {document['text']}")

[note_llm] Un LLM est un modèle entraîné à prédire la suite d'un texte. Il peut produire une réponse plausible qui contient pourtant une erreur.
[note_rag] Le RAG recherche des passages dans des documents et les ajoute au contexte envoyé au LLM. Le RAG ne réentraîne pas le modèle.
[note_embeddings] Un embedding représente un texte par une liste de nombres. On peut comparer ces représentations pour retrouver des textes proches par le sens.


## 4. Première recherche : les mots-clés

Avant d'utiliser des embeddings, on commence avec une méthode simple et visible : compter les mots communs entre la question et chaque document. Ce n'est pas une recherche sémantique ; elle sert de première baseline.

In [2]:
import re

stop_words = {"le", "la", "les", "un", "une", "des", "de", "du", "et", "est", "au", "aux", "dans", "que", "qui", "quoi", "comment", "pourquoi", "il", "elle", "on", "ce", "se", "par", "en", "a", "à"}

def tokenize(text):
    words = re.findall(r"[a-zA-ZÀ-ÿ0-9]+", text.lower())
    return {word for word in words if word not in stop_words}

def retrieve(question, documents, top_k=2):
    question_words = tokenize(question)
    scored_documents = []

    for document in documents:
        document_words = tokenize(document["text"])
        score = len(question_words & document_words)
        scored_documents.append((score, document))

    scored_documents.sort(key=lambda item: item[0], reverse=True)
    return [(score, document) for score, document in scored_documents[:top_k] if score > 0]

question = "Comment mesurer si deux formulations expriment une idée similaire ?"
results = retrieve(question, documents)

for score, document in results:
    print(f"score={score} | [{document['source']}] {document['text']}")

Le score compte les mots partagés, pas la vérité ni la qualité de la réponse. Teste une question formulée autrement : la recherche par mots-clés peut rater un passage pourtant pertinent. C'est une limite attendue de cette première version.

## 5. Construire le contexte envoyé au LLM

Le programme rassemble les passages trouvés et la question dans une consigne. C'est l'étape *Augmented* de RAG.

In [3]:
def build_prompt(question, results):
    if not results:
        context = "Aucun passage pertinent n'a été retrouvé."
    else:
        context = "\n".join(
            f"[{document['source']}] {document['text']}"
            for _, document in results
        )

    return (
        "Réponds à la question en utilisant uniquement les sources ci-dessous. "
        "Si les sources ne permettent pas de répondre, dis-le clairement. "
        "Cite le nom de la source utilisée.\n\n"
        f"Sources :\n{context}\n\n"
        f"Question : {question}\n"
        "Réponse :"
    )
prompt = build_prompt(question, results)
print(prompt)

Réponds à la question en utilisant uniquement les sources ci-dessous. Si les sources ne permettent pas de répondre, dis-le clairement. Cite le nom de la source utilisée.

Sources :
Aucun passage pertinent n'a été retrouvé.

Question : Comment mesurer si deux formulations expriment une idée similaire ?
Réponse :


## 6. Faire générer la réponse

Cette version n'appelle pas de modèle : elle s'arrête volontairement au prompt pour éviter une clé API et rendre visible ce qui est transmis au LLM. Copie le prompt affiché dans un assistant LLM, puis compare sa réponse avec le document source.

Vérifie deux choses : la réponse est-elle soutenue par le passage retrouvé ? La source citée dit-elle vraiment cela ? Une consigne de citation aide à contrôler la réponse, mais ne garantit pas à elle seule qu'elle sera correcte.

## 7. Expériences à faire

1. Remplace `question` par `Que fait un modèle de langage ?` et observe les passages retrouvés.
2. Essaie `Comment représenter un texte pour comparer son sens ?` et note si la recherche par mots-clés trouve la note sur les embeddings.
3. Pose une question dont la réponse n'existe dans aucun document. Le prompt demande au LLM de le reconnaître ; vérifie s'il le fait réellement.
4. Ajoute une quatrième note et relance la recherche.

### À retenir

- Le LLM rédige ; le mécanisme de recherche sélectionne les sources.
- RAG = recherche de passages + ajout au prompt + génération par un LLM.
- La recherche par mots-clés est une baseline simple ; les embeddings peuvent mieux gérer les formulations différentes.
- Il faut vérifier les sources et les réponses, surtout quand une erreur a des conséquences.

Prochaine étape : remplacer la recherche par mots-clés par des embeddings, puis connecter un vrai LLM depuis Python.